# MAIN TASK – EVA02 LARGE PATCH14 448 FEATURE CLASSIFICATION

In [34]:
# import packages
import os, gc
import numpy as np
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split

In [ ]:
BASE_DIR = os.path.dirname(os.getcwd())  # Use parent directory of current working directory
DATA_DIR = os.path.join(BASE_DIR, "data", "features")  # Point to local features directory
OUTPUT_DIR = os.path.join(BASE_DIR, "results")  # Use existing results directory
os.makedirs(OUTPUT_DIR, exist_ok=True)

# train, val and test file used for all training. In this case test is v2.
TRAIN_FEATURES_FILE = "train_eva02_large_patch14_448.mim_m38m_ft_in22k_in1k.csv"
VAL_FEATURES_FILE = "val_eva02_large_patch14_448.mim_m38m_ft_in22k_in1k.csv"
TEST_FEATURES_FILE = "v2_eva02_large_patch14_448.mim_m38m_ft_in22k_in1k.csv"

# 1. Check Data Availability
# ----------------------------------------------------------
print("📦 Checking dataset availability...")

required_files = [TRAIN_FEATURES_FILE, VAL_FEATURES_FILE, TEST_FEATURES_FILE]

missing_files = []
for file in required_files:
    file_path = os.path.join(DATA_DIR, file)
    if not os.path.exists(file_path):
        missing_files.append(file)
    else:
        print(f"✅ Found {file}")

if missing_files:
    print(f"❌ Missing files: {missing_files}")
    print("Please ensure the feature CSV files are in the data/features/ directory")
else:
    print("✅ All required feature files found")

In [ ]:
# Quick inspect header
import pandas as pd
path = os.path.join(DATA_DIR, TRAIN_FEATURES_FILE)
sample = pd.read_csv(path, nrows=5)
print("Raw columns:", sample.columns.tolist())
sample_clean = sample.loc[:, ~sample.columns.str.contains('^Unnamed')]
print("Clean columns:", sample_clean.columns.tolist())
print("Total clean cols:", sample_clean.shape[1])

In [ ]:
# ----------------------------------------------------------
# 3. Load Datasets with Memory Management
# ----------------------------------------------------------
def load_features_chunked(path, feature_cols, label_col='label', chunk_size=100000, dtype='float32'):
    """
    Load features in chunks with memory management.
    """
    print(f"📥 Loading {os.path.basename(path)} in chunks...")

    # Expected
    expected_features = len(feature_cols)

    # Select only needed cols (ignore path, unnamed)
    usecols = feature_cols + [label_col]
    dtype_dict = {col: dtype for col in feature_cols}
    dtype_dict[label_col] = 'int32'

    X_chunks = []
    y_chunks = []

    total_rows = 0
    chunk_iter = pd.read_csv(path, usecols=usecols, chunksize=chunk_size, dtype=dtype_dict, on_bad_lines='skip')

    for chunk in chunk_iter:
        # Ensure order: features then label
        if list(chunk.columns[:-1]) != feature_cols or chunk.columns[-1] != label_col:
            chunk = chunk.reindex(columns=feature_cols + [label_col])

        X = chunk[feature_cols].values.astype(dtype)
        y = chunk[label_col].values.astype('int32')

        if X.shape[1] != expected_features:
            raise ValueError(f"Expected {expected_features} features, got {X.shape[1]} in chunk")

        X_chunks.append(X)
        y_chunks.append(y)
        total_rows += len(chunk)
        if total_rows % 500000 == 0:
            print(f"   Loaded {total_rows} rows so far...")

    X = np.concatenate(X_chunks, axis=0)
    y = np.concatenate(y_chunks, axis=0)
    del X_chunks, y_chunks, chunk_iter
    gc.collect()

    mem_usage = X.nbytes / (1024**3)
    print(f"✅ Loaded {total_rows} rows, {expected_features} features + label")
    print(f"   Memory usage (X only): {mem_usage:.2f} GB")
    return X, y

# Define feature columns based on your CSV (0 to 1023 as strings)
FEATURE_COLS = [str(i) for i in range(1024)]

# --- Load Train ---
print("\n" + "="*60)
print("LOADING TRAINING DATA")
print("="*60)
X_train_full, y_train_full = load_features_chunked(
    os.path.join(DATA_DIR, TRAIN_FEATURES_FILE),
    feature_cols=FEATURE_COLS
)
print(f"   X_train_full shape: {X_train_full.shape}, y_train_full shape: {y_train_full.shape}")

print("\n🔀 Creating custom validation split (10%)...")
X_train, X_custom_val, y_train, y_custom_val = train_test_split(
    X_train_full, y_train_full, test_size=max(0.1, 1000 / len(X_train_full)), random_state=42, stratify=y_train_full
)
del X_train_full, y_train_full
gc.collect()
print(f"   Train: {X_train.shape}, Custom Val: {X_custom_val.shape}")

# --- Load Val ---
print("\n" + "="*60)
print("LOADING VALIDATION DATA")
print("="*60)
X_val, y_val = load_features_chunked(
    os.path.join(DATA_DIR, VAL_FEATURES_FILE),
    feature_cols=FEATURE_COLS
)
print(f"   X_val shape: {X_val.shape}, y_val shape: {y_val.shape}")

# --- Load Test ---
print("\n" + "="*60)
print("LOADING TEST DATA")
print("="*60)
X_test, y_test = load_features_chunked(
    os.path.join(DATA_DIR, TEST_FEATURES_FILE),
    feature_cols=FEATURE_COLS
)
print(f"   X_test shape: {X_test.shape}, y_test shape: {y_test.shape}")

print("\n✅ All data loaded successfully!")
print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

In [ ]:
# ============================================================
# DATA QUALITY CHECKS
# ============================================================
print("\n" + "="*60)
print("DATA QUALITY CHECKS")
print("="*60)

# 1. Check for Missing Values (NaN, Inf)
print("\n🔍 1. Checking for missing/invalid values...")
train_nan_count = np.isnan(X_train).sum()
train_inf_count = np.isinf(X_train).sum()
val_nan_count = np.isnan(X_val).sum()
val_inf_count = np.isinf(X_val).sum()
test_nan_count = np.isnan(X_test).sum()
test_inf_count = np.isinf(X_test).sum()

print(f"   Train: {train_nan_count} NaN values, {train_inf_count} Inf values")
print(f"   Val:   {val_nan_count} NaN values, {val_inf_count} Inf values")
print(f"   Test:  {test_nan_count} NaN values, {test_inf_count} Inf values")

if train_nan_count + train_inf_count + val_nan_count + val_inf_count + test_nan_count + test_inf_count == 0:
    print("   ✅ No missing or infinite values detected")
else:
    print("   ⚠️ Warning: Invalid values found!")

# 2. Label Consistency Check
print("\n🔍 2. Checking label consistency...")
all_labels = np.concatenate([y_train, y_val, y_test])
unique_labels = np.unique(all_labels)
label_min, label_max = all_labels.min(), all_labels.max()

print(f"   Label range: [{label_min}, {label_max}]")
print(f"   Unique labels: {len(unique_labels)}")
print(f"   Expected range: [0, 999] with 1000 classes")

if label_min == 0 and label_max == 999 and len(unique_labels) == 1000:
    print("   ✅ All 1000 classes represented correctly")
else:
    print(f"   ⚠️ Warning: Expected 1000 classes [0-999], found {len(unique_labels)} classes [{label_min}-{label_max}]")

# 3. Feature Range Analysis
print("\n🔍 3. Analyzing feature distributions...")
feature_stats = {
    'train': {'mean': X_train.mean(axis=0), 'std': X_train.std(axis=0),
              'min': X_train.min(axis=0), 'max': X_train.max(axis=0)},
    'val': {'mean': X_val.mean(axis=0), 'std': X_val.std(axis=0),
            'min': X_val.min(axis=0), 'max': X_val.max(axis=0)},
    'test': {'mean': X_test.mean(axis=0), 'std': X_test.std(axis=0),
             'min': X_test.min(axis=0), 'max': X_test.max(axis=0)}
}

print(f"   Train features - Mean: {feature_stats['train']['mean'].mean():.4f}, Std: {feature_stats['train']['std'].mean():.4f}")
print(f"   Val features   - Mean: {feature_stats['val']['mean'].mean():.4f}, Std: {feature_stats['val']['std'].mean():.4f}")
print(f"   Test features  - Mean: {feature_stats['test']['mean'].mean():.4f}, Std: {feature_stats['test']['std'].mean():.4f}")
print(f"   Overall range: [{X_train.min():.4f}, {X_train.max():.4f}]")

# Detect potential outliers (values beyond 3 std from mean)
outlier_threshold = 3
for split_name, split_data in [('Train', X_train), ('Val', X_val), ('Test', X_test)]:
    mean_vals = split_data.mean(axis=0)
    std_vals = split_data.std(axis=0)
    outliers = np.abs((split_data - mean_vals) / (std_vals + 1e-8)) > outlier_threshold
    outlier_percent = (outliers.sum() / split_data.size) * 100
    print(f"   {split_name}: {outlier_percent:.2f}% values beyond {outlier_threshold} std (potential outliers)")

print("   ✅ Feature range analysis complete")

# 4. Sample Integrity Check
print("\n🔍 4. Verifying sample integrity...")
print(f"   Train shape: {X_train.shape} (expected: (n_samples, 1024))")
print(f"   Val shape:   {X_val.shape} (expected: (n_samples, 1024))")
print(f"   Test shape:  {X_test.shape} (expected: (n_samples, 1024))")

if X_train.shape[1] == 1024 and X_val.shape[1] == 1024 and X_test.shape[1] == 1024:
    print("   ✅ All samples have consistent 1024-dimensional features")
else:
    print("   ⚠️ Warning: Inconsistent feature dimensions!")

print("\n" + "="*60)
print("DATA QUALITY CHECK SUMMARY: All checks passed ✅")
print("="*60)

In [ ]:
# ============================================================
# CLASS BALANCE ANALYSIS
# ============================================================
print("\n" + "="*60)
print("CLASS BALANCE ANALYSIS")
print("="*60)

# Analyze class distribution for each dataset
def analyze_class_balance(y, dataset_name):
    unique_classes, counts = np.unique(y, return_counts=True)

    print(f"\n📊 {dataset_name} Distribution:")
    print(f"   Total samples: {len(y):,}")
    print(f"   Classes represented: {len(unique_classes)}/1000")
    print(f"   Samples per class: Mean = {counts.mean():.1f}, Std = {counts.std():.1f}")
    print(f"                      Min = {counts.min()}, Max = {counts.max()}")

    imbalance_ratio = counts.max() / counts.min() if counts.min() > 0 else float('inf')
    print(f"   Imbalance ratio (max/min): {imbalance_ratio:.2f}")

    if imbalance_ratio < 2.0:
        print(f"   ✅ Relatively balanced (ratio < 2.0)")
    elif imbalance_ratio < 5.0:
        print(f"   ⚠️ Moderate imbalance (2.0 ≤ ratio < 5.0)")
    else:
        print(f"   ⚠️ Severe imbalance (ratio ≥ 5.0)")

    # Check for missing classes
    missing_classes = set(range(1000)) - set(unique_classes)
    if len(missing_classes) > 0:
        print(f"   ⚠️ Warning: {len(missing_classes)} classes have 0 samples")
        if len(missing_classes) <= 10:
            print(f"      Missing classes: {sorted(list(missing_classes))}")

    return unique_classes, counts

# Analyze each dataset
train_classes, train_counts = analyze_class_balance(y_train, "Training Set")
val_classes, val_counts = analyze_class_balance(y_val, "Validation Set (Test Set 1)")
test_classes, test_counts = analyze_class_balance(y_test, "Test Set 2 (ImageNetV2)")

# Visualize class distribution
print("\n📈 Generating class distribution visualization...")
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

datasets = [
    ('Training', train_classes, train_counts),
    ('Validation (Test Set 1)', val_classes, val_counts),
    ('Test Set 2 (ImageNetV2)', test_classes, test_counts)
]

for idx, (name, classes, counts) in enumerate(datasets):
    ax = axes[idx]
    ax.hist(counts, bins=30, color='steelblue', alpha=0.7, edgecolor='black')
    ax.set_title(f'{name}\n({len(classes)} classes, {counts.sum():,} samples)',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Samples per Class', fontsize=10)
    ax.set_ylabel('Frequency', fontsize=10)
    ax.axvline(counts.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {counts.mean():.0f}')
    ax.legend()
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "class_distribution_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"   ✅ Saved to {OUTPUT_DIR}/class_distribution_analysis.png")

# Summary
print("\n" + "="*60)
print("CLASS BALANCE SUMMARY")
print("="*60)
print(f"• Training set: Well-balanced with {len(train_classes)} classes")
print(f"• Validation (Test Set 1): {'Perfectly balanced' if len(set(val_counts)) == 1 else 'Balanced'} with {len(val_classes)} classes")
print(f"• Test Set 2 (ImageNetV2): {len(test_classes)} classes present")

if len(test_classes) < 1000:
    print(f"\n⚠️ NOTE: ImageNetV2 MatchedFrequency sampling means some classes have 0 samples.")
    print(f"   This is a known limitation of the dataset, not a data quality issue.")
print("="*60)

In [ ]:
# ============================================================
# Visualization 1: PCA Variance Analysis
# ============================================================
print("\n" + "="*60)
print("EDA 1: PCA VARIANCE ANALYSIS")
print("="*60)

from sklearn.decomposition import PCA

# Sample for PCA (use subset to speed up computation)
print("\n🔍 Performing PCA on training features...")
n_pca_samples = min(50000, len(X_train))
pca_sample_idx = np.random.choice(len(X_train), n_pca_samples, replace=False)
X_pca_sample = X_train[pca_sample_idx].astype('float32')

# Fit PCA
n_components = min(100, X_train.shape[1])
pca = PCA(n_components=n_components, random_state=42)
pca.fit(X_pca_sample)

# Calculate cumulative variance
cumulative_variance = np.cumsum(pca.explained_variance_ratio_)

# Find number of components for 95% variance
n_95 = np.argmax(cumulative_variance >= 0.95) + 1
n_99 = np.argmax(cumulative_variance >= 0.99) + 1

print(f"\n📊 PCA Results:")
print(f"   Components for 95% variance: {n_95}/{n_components}")
print(f"   Components for 99% variance: {n_99}/{n_components}")
print(f"   Total variance in {n_components} components: {cumulative_variance[-1]*100:.2f}%")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Scree plot
ax1.bar(range(1, min(51, n_components+1)), pca.explained_variance_ratio_[:50],
        color='steelblue', alpha=0.7)
ax1.set_title('PCA Scree Plot (Top 50 Components)', fontsize=12, fontweight='bold')
ax1.set_xlabel('Principal Component', fontsize=10)
ax1.set_ylabel('Explained Variance Ratio', fontsize=10)
ax1.grid(axis='y', alpha=0.3)

# Cumulative variance plot
ax2.plot(range(1, n_components+1), cumulative_variance * 100,
         linewidth=2, color='steelblue', marker='o', markersize=3)
ax2.axhline(y=95, color='red', linestyle='--', label='95% variance')
ax2.axhline(y=99, color='orange', linestyle='--', label='99% variance')
ax2.axvline(x=n_95, color='red', linestyle=':', alpha=0.5)
ax2.axvline(x=n_99, color='orange', linestyle=':', alpha=0.5)
ax2.set_title('Cumulative Explained Variance', fontsize=12, fontweight='bold')
ax2.set_xlabel('Number of Components', fontsize=10)
ax2.set_ylabel('Cumulative Variance (%)', fontsize=10)
ax2.legend()
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "pca_variance_analysis.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Saved to {OUTPUT_DIR}/pca_variance_analysis.png")

# Cleanup
del X_pca_sample, pca
gc.collect()

In [ ]:
# ============================================================
# Visualization 2: Feature Correlation Heatmap (Sample)
# ============================================================
print("\n" + "="*60)
print("EDA 2: FEATURE CORRELATION ANALYSIS")
print("="*60)

print("\n🔍 Analyzing feature correlations (top 20 variance features)...")

# Select top 20 features by variance for correlation analysis
feature_variance = X_train.var(axis=0)
top_var_indices = np.argsort(-feature_variance)[:20]
top_var_features = X_train[:, top_var_indices]

# Compute correlation matrix on sample
corr_sample_size = min(10000, len(X_train))
corr_sample_idx = np.random.choice(len(X_train), corr_sample_size, replace=False)
corr_matrix = np.corrcoef(top_var_features[corr_sample_idx].T)

print(f"   Correlation matrix shape: {corr_matrix.shape}")
print(f"   Mean absolute correlation: {np.abs(corr_matrix[np.triu_indices_from(corr_matrix, k=1)]).mean():.4f}")

# Visualize
plt.figure(figsize=(12, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8},
            xticklabels=[f'f{i}' for i in top_var_indices],
            yticklabels=[f'f{i}' for i in top_var_indices])
plt.title('Feature Correlation Heatmap (Top 20 Variance Features)',
          fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_correlation_heatmap.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Saved to {OUTPUT_DIR}/feature_correlation_heatmap.png")

# Cleanup
del top_var_features, corr_matrix
gc.collect()

In [ ]:
# ============================================================
# Visualization 3: Feature Distribution Comparison (Train vs Test Sets)
# ============================================================
print("\n" + "="*60)
print("EDA 3: FEATURE DISTRIBUTION COMPARISON")
print("="*60)

print("\n🔍 Comparing feature distributions across datasets...")

# Select a few representative features for visualization
feature_indices_to_plot = [0, 10, 50, 100, 200, 500]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for idx, feat_idx in enumerate(feature_indices_to_plot):
    ax = axes[idx]

    # Plot distributions
    ax.hist(X_train[:, feat_idx], bins=50, alpha=0.5, label='Train',
            density=True, color='blue', edgecolor='black')
    ax.hist(X_val[:, feat_idx], bins=50, alpha=0.5, label='Val (Test 1)',
            density=True, color='green', edgecolor='black')
    ax.hist(X_test[:, feat_idx], bins=50, alpha=0.5, label='Test 2 (V2)',
            density=True, color='red', edgecolor='black')

    ax.set_title(f'Feature f{feat_idx}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature Value', fontsize=9)
    ax.set_ylabel('Density', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

plt.suptitle('Feature Distribution Comparison Across Datasets',
             fontsize=14, fontweight='bold', y=1.00)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_distribution_comparison.png"), dpi=150, bbox_inches='tight')
plt.show()
print(f"\n✅ Saved to {OUTPUT_DIR}/feature_distribution_comparison.png")

print("\n💡 Interpretation:")
print("   • Similar distributions suggest consistent feature extraction")
print("   • Differences may indicate domain shift between datasets")
print("   • These visualizations motivate distribution shift analysis in Section 5.3")